# Clase 045 — SQL desde Python

**Parte 0** · sqlite3 + SQLAlchemy + DuckDB.

> 🎯 Las 3 formas que vas a usar en producción. Parametrización para evitar injection.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

# DataFrame demo
df = pd.DataFrame({
    'cliente_id': range(1, 11),
    'nombre': [f'Cliente {i}' for i in range(1, 11)],
    'pais': rng.choice(['ES', 'CL', 'MX'], 10),
    'monto': rng.uniform(50, 500, 10).round(2),
})
print(df.head())

## 1️⃣ `sqlite3` stdlib

Flow básico: `connect` → `cursor` → `execute(sql, params)` → `fetchall()`.

In [ ]:
con = sqlite3.connect(':memory:')
cur = con.cursor()
cur.execute('CREATE TABLE clientes (id INTEGER, nombre TEXT, pais TEXT, monto REAL)')

# executemany con tuples para insertar varios
datos = [(row.cliente_id, row.nombre, row.pais, row.monto) for row in df.itertuples()]
cur.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', datos)
con.commit()

# Consulta con placeholder ?
for row in cur.execute('SELECT * FROM clientes WHERE pais = ? AND monto > ?', ('ES', 200)):
    print(row)

## 2️⃣ ⚠️ NUNCA concatenes SQL

```python
# ❌ MAL — vulnerable a injection
user_input = "ES'; DROP TABLE clientes; --"
cur.execute(f"SELECT * FROM clientes WHERE pais = '{user_input}'")  # ¡catástrofe!

# ✅ BIEN — placeholder seguro
cur.execute('SELECT * FROM clientes WHERE pais = ?', (user_input,))
```

El driver escapa el valor automáticamente. Es **la** regla de seguridad de SQL desde código.

## 3️⃣ `pd.read_sql` y `df.to_sql`

Pandas tiene pasarela bidireccional:

In [ ]:
# DataFrame → tabla
df.to_sql('clientes_pd', con, if_exists='replace', index=False)

# Tabla → DataFrame
result = pd.read_sql('SELECT pais, AVG(monto) AS avg_m FROM clientes_pd GROUP BY pais', con)
print(result)

## 4️⃣ SQLAlchemy — backend-agnostic

```python
from sqlalchemy import create_engine

# URLs por motor:
#   sqlite:///archivo.db           — SQLite local
#   sqlite:///:memory:             — SQLite en memoria
#   postgresql://user:pw@host/db   — Postgres
#   mysql+pymysql://user:pw@host/db— MySQL

engine = create_engine('sqlite:///:memory:')
df.to_sql('clientes', engine, if_exists='replace', index=False)
result = pd.read_sql('SELECT * FROM clientes WHERE monto > 200', engine)
```

Ventaja: cambias 1 string en el engine y migras de SQLite a Postgres sin tocar el resto del código.

In [ ]:
try:
    from sqlalchemy import create_engine
    engine = create_engine('sqlite:///:memory:')
    df.to_sql('cl', engine, if_exists='replace', index=False)
    print(pd.read_sql('SELECT pais, COUNT(*) c FROM cl GROUP BY pais', engine))
except ImportError:
    print('Instala SQLAlchemy: pip install sqlalchemy')

## 5️⃣ DuckDB — SQL sobre DataFrames y archivos

DuckDB es como SQLite pero **columnar** (optimizado para analytics) y **lee CSV/Parquet directamente** sin cargar a memoria:

```python
import duckdb
# Sobre DataFrame en memoria
duckdb.query('SELECT species, AVG(body_mass_g) FROM df GROUP BY species').df()

# Sobre CSV directo (sin pandas)
duckdb.query("SELECT species, COUNT(*) FROM 'penguins.csv' GROUP BY species").df()

# Sobre Parquet (mucho más rápido)
duckdb.query("SELECT * FROM 'datos.parquet' LIMIT 100").df()
```

In [ ]:
try:
    import duckdb
    # SQL sobre nuestro DataFrame
    result = duckdb.query('''
        SELECT pais,
               COUNT(*)       AS n,
               AVG(monto)     AS avg_monto,
               MAX(monto)     AS max_monto
        FROM df
        GROUP BY pais
        ORDER BY avg_monto DESC
    ''').df()
    print(result.round(2))
except ImportError:
    print('Instala DuckDB: pip install duckdb')

## 🧭 Cuándo cada uno

| Tool | Caso |
|---|---|
| `sqlite3` stdlib | Demos, tests, BDs locales pequeñas, scripts one-shot |
| SQLAlchemy | Producción con PostgreSQL/MySQL; ORMs, migraciones |
| DuckDB | Análisis ad-hoc sobre CSV/Parquet, EDA rápido con SQL |

## ✅ Checklist

- [ ] Uso placeholders `?` en sqlite3 (NUNCA concatenar)
- [ ] Sé pasarela `df.to_sql` / `pd.read_sql`
- [ ] Conozco SQLAlchemy para producción
- [ ] Uso DuckDB para SQL sobre CSV/Parquet sin cargar a pandas
- [ ] Sé qué tool elegir según contexto

## 📝 Homework

Ver `README.md`. 3 backends mismo análisis + demo injection.

## 📖 Definiciones y características

**`sqlite3` (stdlib)**

Driver Python para SQLite. Sin dependencias externas. Patrón: `connect(...)` → `cursor()` → `execute(sql, params)` → `fetchall()`.

**Parameterized query (`?` o `:name`)**

Placeholder donde el driver substituye valores escapados. **Única forma segura** de pasar datos de usuario — previene SQL injection.

**SQLAlchemy**

Toolkit ORM + Core para Python. Backend-agnostic: cambias el URL del engine y migras entre SQLite/Postgres/MySQL sin tocar queries. `create_engine('postgresql://...')`.

**DuckDB**

OLAP DB embebida. SQL sobre DataFrames pandas (`duckdb.query('SELECT ... FROM df')`) y archivos CSV/Parquet directos (`FROM 'data.csv'`). Mucho más rápido que sqlite3 para analytics.

**SQL injection**

Inyección de SQL malicioso via concatenación de strings con input de usuario. **Prevención**: SIEMPRE parameterized queries, nunca f-string con valores externos.

**`pd.read_sql` / `df.to_sql`**

Pasarela pandas↔BD. Acepta connection o engine. `read_sql_query` para queries complejas; `read_sql_table` para tablas completas.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `OperationalError: no such table: X` después de insert | Falta `con.commit()`. sqlite3 no auto-commit. **Fix**: `con.commit()` tras INSERT/UPDATE/DELETE, o `con = sqlite3.connect(':memory:', isolation_level=None)` para auto-commit. |
| Concatené input de usuario en query y funcionó | Hasta que el usuario malicioso prueba `'; DROP TABLE clientes; --`. **Fix**: NUNCA `f"SELECT * FROM x WHERE id={user}"`. Siempre `(?, ?)` placeholders. |
| SQLAlchemy 2.0 — `Engine.execute` no existe | API cambió: ahora `with engine.connect() as con: con.execute(text('SELECT ...'))`. Tutorial oficial actualizado. |
| DuckDB lee CSV pero pierde tipos | Pandas adivina mejor. **Fix**: `duckdb.read_csv('x.csv', dtype={'col': 'INT'})` o convierte después. |
| `pd.read_sql` lento con N grande | Driver Python carga todo a Python. **Fix**: `chunksize=10000` itera por bloques, o usa DuckDB directo sobre BD (cuando aplica). |

## ❓ Preguntas frecuentes

**❓ ¿sqlite3, SQLAlchemy o DuckDB?**

**sqlite3** para demos/tests/scripts locales. **SQLAlchemy** para producción con Postgres/MySQL. **DuckDB** para EDA sobre CSV/Parquet sin servidor — el más rápido para analytics.

**❓ ¿`pd.read_sql` es seguro contra injection?**

Sí si pasas params: `read_sql('SELECT * FROM t WHERE x=:val', con, params={'val': user_input})`. No si concatenas strings.

**❓ ¿ORM (SQLAlchemy declarative) o queries directas?**

ORM cuando el modelo se usa en muchas partes (web app con N modelos). Queries directas para análisis ad-hoc. Pueden coexistir.

**❓ ¿DuckDB sobre Parquet vs sobre pandas?**

**Parquet directo** es más rápido (no carga a Python). **Pandas** cuando ya tienes el DataFrame en memoria. DuckDB es smart: optimiza ambos casos.

**❓ ¿Cerrar conexión manualmente?**

Usa context manager: `with sqlite3.connect(...) as con: ...` o `con.close()` en `finally`. Conexiones dejadas abiertas consumen handles del OS.

## 🔗 Referencias

- [sqlite3](https://docs.python.org/3/library/sqlite3.html)
- [SQLAlchemy](https://docs.sqlalchemy.org/en/20/tutorial/)
- [DuckDB Python](https://duckdb.org/docs/api/python/overview)

➡️ **Siguiente:** [046 — MongoDB](../046-nosql-mongodb-con-pymongo/README.md)

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables sin internet** de los ejercicios del README. Cada bloque incluye comentarios y comprobaciones (`assert`/`print`). Intenta resolverlos tú antes de mirar.

In [ ]:
import sqlite3, pandas as pd, numpy as np
rng = np.random.default_rng(42)
df = pd.DataFrame({'cliente_id': range(1, 11),
                   'nombre': [f'Cliente {i}' for i in range(1, 11)],
                   'pais': rng.choice(['ES', 'CL', 'MX'], 10),
                   'monto': rng.uniform(50, 500, 10).round(2)})
print(df.head(3).to_string(index=False))

**Ejercicio 1 — sqlite3 con placeholders.** `executemany` + `?`, y demo del bug al concatenar.

In [ ]:
con = sqlite3.connect(':memory:'); cur = con.cursor()
cur.execute('CREATE TABLE clientes (id INTEGER, nombre TEXT, pais TEXT, monto REAL)')
datos = [(r.cliente_id, r.nombre, r.pais, r.monto) for r in df.itertuples()]
cur.executemany('INSERT INTO clientes VALUES (?, ?, ?, ?)', datos)   # tuples + placeholders
con.commit()
rows = cur.execute('SELECT nombre FROM clientes WHERE pais = ? AND monto > ?',
                   ('ES', 200)).fetchall()
print('ES con monto>200:', [x[0] for x in rows])
# BUG si concatenas: una comilla en la entrada altera la lógica ("OR '1'='1" -> todo).
malicioso = "ES' OR '1'='1"
cur.execute(f"SELECT * FROM clientes WHERE pais = '{malicioso}'")
n = len(cur.fetchall())
print(f'Concatenando, el filtro devolvió {n} filas (el OR 1=1 se coló => TODAS).')
print('Lección: SIEMPRE placeholders ?, NUNCA f-strings dentro del SQL.')

**Ejercicio 2 — `df.to_sql` y `pd.read_sql`.** Ida y vuelta DataFrame ↔ SQLite.

In [ ]:
df.to_sql('clientes_pd', con, if_exists='replace', index=False)
back = pd.read_sql('SELECT pais, ROUND(AVG(monto), 2) AS avg_m FROM clientes_pd GROUP BY pais', con)
print(back.to_string(index=False))
assert set(back.pais).issubset({'ES', 'CL', 'MX'})
print('OK: DataFrame -> tabla SQLite -> DataFrame de vuelta')

**Ejercicio 3 — SQLAlchemy engine.** `create_engine` + `pd.read_sql` sobre una Connection.

In [ ]:
from sqlalchemy import create_engine, text
engine = create_engine('sqlite:///:memory:')
with engine.begin() as conn:                 # en pandas 3.x se pasa una Connection
    df.to_sql('cl', conn, if_exists='replace', index=False)
    r = pd.read_sql(text('SELECT pais, COUNT(*) AS c FROM cl GROUP BY pais'), conn)
print(r.to_string(index=False))
print('SQLAlchemy da una URL por motor: cambia sqlite:/// por postgresql:// sin tocar el resto.')

**Ejercicio 4 — DuckDB sobre DataFrame.** SQL directo sobre una variable pandas.

In [ ]:
import duckdb
penguins_like = pd.DataFrame({'species': ['A', 'A', 'B', 'B', 'C'],
                              'body_mass_g': [3700, 3900, 5000, 5200, 3600]})
r = duckdb.query('''SELECT species, AVG(body_mass_g) AS avg_mass
                    FROM penguins_like GROUP BY species ORDER BY species''').df()
print(r.to_string(index=False))
assert list(r.species) == ['A', 'B', 'C']
print('DuckDB consulta el DataFrame en memoria por su nombre de variable (sin copiarlo).')

**Ejercicio 5 — DuckDB sobre CSV.** `FROM 'archivo.csv'` sin cargar a pandas.

In [ ]:
import os, tempfile
csv_path = os.path.join(tempfile.mkdtemp(), 'penguins.csv')
penguins_like.to_csv(csv_path, index=False)
r = duckdb.query(f"SELECT species, AVG(body_mass_g) AS avg_mass "
                 f"FROM '{csv_path}' GROUP BY species ORDER BY species").df()
print(r.to_string(index=False))
print('DuckDB lee el CSV directamente en el FROM, sin pd.read_csv previo.')